In [0]:
# Celda 1
dbutils.widgets.removeAll()

In [0]:
# Celda 2
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import Window as W

In [0]:
# Celda 3
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
# Celda 4
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# Celda 5
df_catalog_vs_bs = spark.table(f"{catalogo}.{esquema_source}.catalog_vs_bestsellers")
df_catalog = spark.table(f"{catalogo}.{esquema_source}.books_catalog_transformed")
df_country = spark.table(f"{catalogo}.{esquema_source}.country_reading_transformed")

In [0]:
# Celda 6
df_top_authors = df_catalog_vs_bs.groupBy(col("bestseller_author").alias("author")).agg(
    F.count("*").alias("total_bestseller_books"),
    F.round(F.avg("catalog_rating"), 2).alias("avg_user_rating"),
    F.countDistinct("year").alias("years_active")
).orderBy(F.desc("total_bestseller_books"))

In [0]:
# Celda 7
df_top_authors.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.top_authors")

In [0]:
# Celda 8
df_top_books = df_catalog_vs_bs.groupBy(
    col("bestseller_name").alias("name"),
    col("bestseller_author").alias("author"),
    col("bestseller_genre").alias("genre")
).agg(
    F.countDistinct("year").alias("years_as_bestseller"),
    F.round(F.avg("catalog_rating"), 2).alias("avg_user_rating")
).select(
    "name", "author", "years_as_bestseller", "avg_user_rating", "genre"
).orderBy(F.desc("years_as_bestseller"))

In [0]:
# Celda 9
df_top_books.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.top_books")

In [0]:
# Celda 10
df_genre_dist = df_catalog.filter(col("primary_genre") != "").groupBy(
    col("primary_genre").alias("genre")
).agg(
    F.count("*").alias("book_count"),
    F.round(F.avg("rating"), 2).alias("avg_rating")
).orderBy(F.desc("book_count"))

In [0]:
# Celda 11
df_genre_dist.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.genre_distribution")

In [0]:
# Celda 12
genre_window = W.partitionBy("primary_genre").orderBy(F.desc("rating"))

df_top_by_genre = df_catalog.filter(col("primary_genre") != "") \
    .withColumn("rank", F.rank().over(genre_window)) \
    .filter(col("rank") <= 5) \
    .select(
        col("primary_genre").alias("genre"),
        col("rank"),
        col("title"),
        col("author"),
        col("rating")
    ).orderBy("genre", "rank")

In [0]:
# Celda 13
df_top_by_genre.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.top_books_by_genre")

In [0]:
# Celda 14
df_country_golden = df_country.select(
    col("country"),
    col("books_read_annually").alias("reading_index"),
    col("reading_rank").alias("rank")
)

In [0]:
# Celda 15
df_country_golden.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.country_reading")